In [ ]:
# %% [markdown]
# # Exploratory Data Analysis - Phishing Email Detection
# 
# This notebook explores the phishing email dataset and visualizes patterns.

# %% Setup and Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))
from src.data_collection import DataCollector
from src.preprocessing import DataPreprocessor

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# %% Load Data
collector = DataCollector()
df = collector.load_dataset()
print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts())

# %% Visualize Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
ax = axes[0]
df['label'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Email Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Email Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

# Pie chart
ax = axes[1]
df['label'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax, 
                                  colors=['green', 'red'], explode=[0, 0.05])
ax.set_title('Class Proportions', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig('../results/visualizations/class_distribution.png', dpi=100)
plt.show()

# %% Analyze Email Lengths
df['length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot of lengths by class
ax = axes[0, 0]
df.boxplot(column='length', by='label', ax=ax)
ax.set_title('Email Length Distribution by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Character Count')

# Box plot of word counts
ax = axes[0, 1]
df.boxplot(column='word_count', by='label', ax=ax)
ax.set_title('Word Count Distribution by Class', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Word Count')

# Histogram of lengths
ax = axes[1, 0]
for label in df['label'].unique():
    subset = df[df['label'] == label]
    ax.hist(subset['length'], alpha=0.5, label=label, bins=20)
ax.set_title('Email Length Histogram', fontweight='bold')
ax.set_xlabel('Character Count')
ax.set_ylabel('Frequency')
ax.legend()

# Histogram of word counts
ax = axes[1, 1]
for label in df['label'].unique():
    subset = df[df['label'] == label]
    ax.hist(subset['word_count'], alpha=0.5, label=label, bins=20)
ax.set_title('Word Count Histogram', fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.savefig('../results/visualizations/length_analysis.png', dpi=100)
plt.show()

# %% Preprocess data for word clouds
preprocessor = DataPreprocessor()
df_processed = preprocessor.preprocess_dataset(df.copy())

# Separate by class
phishing_text = ' '.join(df_processed[df_processed['label']=='phishing']['cleaned_text'])
legitimate_text = ' '.join(df_processed[df_processed['label']=='legitimate']['cleaned_text'])

# %% Create Word Clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Phishing word cloud
ax = axes[0]
wordcloud_phishing = WordCloud(width=800, height=400, 
                               background_color='black', 
                               colormap='Reds',
                               max_words=100).generate(phishing_text)
ax.imshow(wordcloud_phishing, interpolation='bilinear')
ax.set_title('Phishing Emails - Common Words', fontsize=16, fontweight='bold')
ax.axis('off')

# Legitimate word cloud
ax = axes[1]
wordcloud_legit = WordCloud(width=800, height=400, 
                            background_color='white', 
                            colormap='Greens',
                            max_words=100).generate(legitimate_text)
ax.imshow(wordcloud_legit, interpolation='bilinear')
ax.set_title('Legitimate Emails - Common Words', fontsize=16, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('../results/visualizations/wordcloud_comparison.png', dpi=100)
plt.show()

# %% Analyze Feature Differences
# Extract features for all emails
features_list = []
for idx, row in df.iterrows():
    features = preprocessor.text_preprocessor.extract_features(row['text'])
    features['label'] = row['label']
    features_list.append(features)

features_df = pd.DataFrame(features_list)

# Visualize feature differences
feature_cols = ['url_count', 'email_count', 'urgent_keyword_count', 
                'exclamation_count', 'all_caps_count']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feature in enumerate(feature_cols):
    ax = axes[i]
    
    # Box plot
    features_df.boxplot(column=feature, by='label', ax=ax)
    ax.set_title(f'{feature.replace("_", " ").title()}', fontweight='bold')
    ax.set_xlabel('')
    
    # Add statistics
    phishing_mean = features_df[features_df['label']=='phishing'][feature].mean()
    legit_mean = features_df[features_df['label']=='legitimate'][feature].mean()
    
    print(f"{feature}:")
    print(f"  Phishing mean: {phishing_mean:.2f}")
    print(f"  Legitimate mean: {legit_mean:.2f}")
    print()

# Hide empty subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig('../results/visualizations/feature_comparison.png', dpi=100)
plt.show()

# %% Correlation Analysis
# Prepare data for correlation
corr_data = features_df[feature_cols + ['label']].copy()
corr_data['label_encoded'] = (corr_data['label'] == 'phishing').astype(int)
corr_data = corr_data.drop('label', axis=1)

# Calculate correlation
corr_matrix = corr_data.corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/visualizations/correlation_matrix.png', dpi=100)
plt.show()

# %% Summary Statistics
print("\n" + "="*60)
print("DATASET SUMMARY STATISTICS")
print("="*60)

print(f"\nTotal emails: {len(df)}")
print(f"Phishing emails: {len(df[df['label']=='phishing'])}")
print(f"Legitimate emails: {len(df[df['label']=='legitimate'])}")

print("\nText Length Statistics:")
print(df.groupby('label')['length'].describe())

print("\nWord Count Statistics:")
print(df.groupby('label')['word_count'].describe())

print("\nFeature Statistics (Phishing):")
print(features_df[features_df['label']=='phishing'][feature_cols].describe())

print("\nFeature Statistics (Legitimate):")
print(features_df[features_df['label']=='legitimate'][feature_cols].describe())

print("\n" + "="*60)
print("EXPLORATORY ANALYSIS COMPLETE")
print("="*60)